In [ ]:
%load_ext autoreload
%autoreload 2
import os
import sys
import pandas as pd

from torch.backends import cudnn
from tqdm import tqdm

# enforce more deterministic behavior
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
cudnn.deterministic = True
cudnn.benchmark = False

sys.path.append("../..")
from pneuma_seeker.core.ir_system.main import IRSystem
from pneuma_seeker.core.ir_system.data_model import RetrieverType
from pneuma_seeker.utils.config import Config
from pneuma_seeker.utils.logger import setup_logger
from pneuma_seeker.core.ir_system.data_model import AbstractDocument
from pneuma_seeker.core.ir_system.data_model import Table, TableContext
from pneuma_seeker.model.interface.model_factory import get_embed_model, get_llm

In [ ]:
INDEXING_ARCHEOLOGY = False
INDEXING_BIOMEDICAL = False
INDEXING_ENVIRONMENT = False
INDEXING_TAG = False
INDEXING_BUYSITE = False

In [ ]:
config = Config("../../.env")
llm_path = "../model/weight/qwen3-8b"
embed_model_path = "../model/weight/bge-base"
logger = setup_logger(log_path=os.path.join(".", "log"))

In [ ]:
def index_dataset(dataset_name: str, metadata_available = False):
    DATASET_DIR = f"../../data_src/{dataset_name}/dataset"
    ACTUAL_PATH = os.path.join("..", DATASET_DIR)

    documents: list[AbstractDocument] = []
    dataset = os.listdir(ACTUAL_PATH)
    for table_name in tqdm(dataset, desc="Loading dataset..."):
        table = pd.read_csv(f"{ACTUAL_PATH}/{table_name}")
        documents.append(
            Table(
                doc_id=f"{DATASET_DIR}/{table_name}",
                retriever_type=RetrieverType.PNEUMA,
                content=table,
                metadata={
                    "table_name": f"{DATASET_DIR}/{table_name}",
                    "dataset_name": dataset_name
                }
            )
        )
    
    if metadata_available:
        dataset_metadata = pd.read_csv(f"../../../data_src/{dataset_name}/metadata.csv")
        for _, row in tqdm(dataset_metadata.iterrows(), desc="Loading metadata..."):
            table_name = row["table_name"]
            description = row["description"]
            documents.append(
                TableContext(
                    doc_id=f"context_{DATASET_DIR}/{table_name}",
                    retriever_type=RetrieverType.PNEUMA,
                    content=description,
                    metadata={
                        "table_name": f"{DATASET_DIR}/{table_name}",
                        "dataset_name": dataset_name,
                        "type": "description",
                    }
                )
            )

    ir_sys = IRSystem(
        get_llm(llm_path, config)(llm_path, config, logger),
        get_embed_model()(embed_model_path, config, logger),
        logger,
    )
    ir_sys.index_documents(
        RetrieverType.PNEUMA,
        documents
    )

In [ ]:
if INDEXING_ARCHEOLOGY:
    index_dataset("archeology", True)
if INDEXING_BIOMEDICAL:
    index_dataset("biomedical", True)
if INDEXING_ENVIRONMENT:
    index_dataset("environment", True)
if INDEXING_TAG:
    index_dataset("tag", True)
if INDEXING_BUYSITE:
    index_dataset("buysite", True)

In [ ]:
# Extra: Processing for TAG data
# for topic in os.listdir(DATASET_DIR):
#     topic_path = f"dataset/{topic}"
#     if os.path.isdir(topic_path):
#         for table_fname in os.listdir(topic_path):
#             original_table_path = f"{topic_path}/{table_fname}"
#             appended_table_path = f"{topic_path}/{topic}_{table_fname}"
#             print(f"=> {original_table_path} => {appended_table_path}")
#             os.rename(original_table_path, appended_table_path)
# Future-TODO: don't forget to move the tables outside (manually for now)